> **Repository copy:** code only, with execution outputs removed. Run only in an authorized All of Us Workbench. Generated tables, audits, plots, HTML exports, and individual-level intermediate files stay in that environment. See `README.md` and `docs/provenance.md` for setup and version details.


# v2.3 companion — formal proportional-hazards test (R, `survival::cox.zph`)

Run **after** `rao_tmvl_testing_v2_3.ipynb`, which writes `internal/stroke_ph_data_INTERNAL.csv` (individual-level; keep inside the Workbench).
Open this notebook with the **R** kernel and use *Kernel → Restart Kernel and Run All Cells*. Save the executed notebook and export it to HTML.

Outputs (all in `results/`; retain inside the authorized Workbench pending disclosure review):
- `T6b_cox_zph_tests.csv` — covariate-level and global tests, KM and log time transforms, 1- and 3-year horizons.
- `T6b_rao_beta_t_*.png` — smoothed time-varying coefficient for RAO with 95% band (residual points omitted).
- `T6d_piecewise_HR.csv` — adjusted RAO-vs-TMVL HR for days 0–30 and 31–1095 from one model, plus a test of whether they differ.

Counts of events ≤20 in either group suppress the corresponding estimate.

In [ ]:
suppressPackageStartupMessages(library(survival))
MIN_CELL <- 20
dir.create("results", showWarnings = FALSE)
d <- read.csv("internal/stroke_ph_data_INTERNAL.csv")
covs <- setdiff(names(d), c("time", "status"))
cat("Covariates:", paste(covs, collapse = ", "), "\n")
stopifnot("rao" %in% covs)
fmt <- function(b, se) sprintf("%.2f (%.2f-%.2f)", exp(b), exp(b - 1.96 * se), exp(b + 1.96 * se))
horizon_data <- function(h) {
  x <- d
  x$ev <- as.integer(x$status == 1 & x$time <= h + 0.5)
  x$t <- pmin(x$time, h + 0.5)
  x
}
enough <- function(x) sum(x$ev[x$rao == 1]) > MIN_CELL && sum(x$ev[x$rao == 0]) > MIN_CELL
form <- function(lhs, terms) as.formula(paste(lhs, "~", paste(terms, collapse = " + ")))

## Formal test: `cox.zph` (covariate-level and global)

In [ ]:
rows <- list()
for (h in c(365, 1095)) {
  x <- horizon_data(h)
  if (!enough(x)) { cat("Horizon", h, ": suppressed (<=20 events in a group)\n"); next }
  fit <- coxph(form("Surv(t, ev)", covs), data = x, ties = "breslow")
  cat("\n=== Horizon", h, "days: adjusted HR, RAO vs TMVL =", fmt(coef(fit)["rao"], sqrt(vcov(fit)["rao", "rao"])), "\n")
  for (tr in c("km", "log")) {
    z <- cox.zph(fit, transform = tr)
    tab <- as.data.frame(z$table)
    tab$term <- rownames(tab); tab$transform <- tr; tab$horizon_days <- h
    rows[[length(rows) + 1]] <- tab
    print(round(z$table, 4))
    if (tr == "km") {
      png(sprintf("results/T6b_rao_beta_t_%dd.png", h), width = 900, height = 650, res = 130)
      plot(z[which(rownames(z$table) == "rao")], resid = FALSE, se = TRUE, xlab = "Days since index (KM-transformed axis)",
           ylab = "Beta(t) for RAO vs TMVL", main = sprintf("Time-varying RAO coefficient, %d-day horizon", h))
      abline(h = coef(fit)["rao"], lty = 2); abline(h = 0, lty = 3)
      dev.off()
    }
  }
}
if (length(rows)) {
  out <- do.call(rbind, rows)[, c("horizon_days", "transform", "term", "chisq", "df", "p")]
  out$chisq <- round(out$chisq, 3); out$p <- signif(out$p, 3)
  write.csv(out, "results/T6b_cox_zph_tests.csv", row.names = FALSE)
  print(out, row.names = FALSE)
}

## Piecewise model: separate RAO-vs-TMVL HRs for days 0–30 and 31–1095

In [ ]:
x <- horizon_data(1095)
sp <- survSplit(Surv(t, ev) ~ ., data = x, cut = 30.5, episode = "period")   # period 1 = days 0-30, 2 = 31-1095
sp$rao_early <- sp$rao * (sp$period == 1)
sp$rao_late  <- sp$rao * (sp$period == 2)
other <- setdiff(covs, "rao")
ev <- function(p, g) sum(sp$ev[sp$period == p & sp$rao == g])
ok_early <- ev(1, 1) > MIN_CELL && ev(1, 0) > MIN_CELL
ok_late  <- ev(2, 1) > MIN_CELL && ev(2, 0) > MIN_CELL
fit_pw <- coxph(form("Surv(tstart, t, ev)", c("rao_early", "rao_late", other)), data = sp, ties = "breslow")
# same model, parameterized so one coefficient is the log ratio of late to early HR
sp$rao_any <- sp$rao
fit_int <- coxph(form("Surv(tstart, t, ev)", c("rao_any", "rao_late", other)), data = sp, ties = "breslow")
b <- coef(fit_pw); se <- sqrt(diag(vcov(fit_pw)))
p_diff <- summary(fit_int)$coefficients["rao_late", "Pr(>|z|)"]
pw <- data.frame(
  period = c("Days 0-30", "Days 31-1095", "Difference (late vs early)"),
  adjusted_HR_RAO_vs_TMVL = c(if (ok_early) fmt(b["rao_early"], se["rao_early"]) else "suppressed",
                              if (ok_late)  fmt(b["rao_late"],  se["rao_late"])  else "suppressed",
                              if (ok_early && ok_late) sprintf("p = %.3g", p_diff) else "suppressed"))
write.csv(pw, "results/T6d_piecewise_HR.csv", row.names = FALSE)
print(pw, row.names = FALSE)

In [ ]:
sessionInfo()$otherPkgs$survival$Version
cat("Done. Save (File -> Save) and export to HTML.\n")